In [ ]:
from IPython.core.display import display
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random
import seaborn as sns
import sklearn
import sklearn.base
import sklearn.compose
import sklearn.inspection
import sklearn.model_selection
import sklearn.linear_model
import sklearn.pipeline
import sklearn.preprocessing
import scipy.stats
import typing

In [ ]:
!git clone https://github.com/nzmonzmp/dataset-ames.git

# Régression linéaire

## Régression linéaire univariée

### Introduction

La régression linéaire est très intéressante à coder soi-même : elle n'est pas trop complexe mais permet de toucher à beaucoup de concepts du machine learning.

Nous allons continuer notre travail sur les données de prix de maisons.

Passons en python quelques secondes pour charger le dataset :

In [ ]:
train_df = pd.read_csv("dataset-ames/train.csv", index_col="Id")

In [ ]:
display(train_df)

### Colonnes utilisées

Nous utiliserons dans ce TP seulement une caractéristique : `GrLivArea`, et `SalePrice` comme target.

- *Plottez `GrLivArea` en fonction de `SalePrice`* en utilisant [`sns.jointplot`](https://seaborn.pydata.org/generated/seaborn.jointplot.html?highlight=jointplot#seaborn.jointplot)
- *Définissez puis normalisez `x` et `y`*. Pour la normalisation vous pouvez :
  - soit utiliser [`sklearn.preprocessing.StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
  - soit calculer vous même la normalisation $$\mathbf{x}_{norm}=\frac{\mathbf{x} - \bar{\mathbf{x}}}{\sigma_\mathbf{x}}$$

In [ ]:
# sns.jointplot(???)
# plt.show()

In [ ]:
# Normalisation de x
# Votre code ici

# Normalisation de y
# Votre code ici

#### Solution

In [ ]:
sns.jointplot(x="GrLivArea", y="SalePrice", data=train_df)
plt.show()

In [ ]:
### Standardize x
x_scaler = sklearn.preprocessing.StandardScaler()
x = x_scaler.fit_transform(train_df[["GrLivArea"]])[:, 0]

### Standardize y
y_scaler = sklearn.preprocessing.StandardScaler()
y = y_scaler.fit_transform(train_df[["SalePrice"]])[:, 0]

### Exemple de Régression Linéaire
Pour comprendre l'évolution jointe du prix de vente et de la surface d'une maison, on va se limiter à une hypothèse forte : on peut trouver une fonction affine qui modélise correctement la relation entre les entrées et les sorties. Pour rappel, une fonction affine est de la forme $f(x) = ax + b$.

Géométriquement, on pourra représenter cette fonction par une droite.

Par exemple, ici, on aimerait trouver l'hypothèse suivante :

In [ ]:
sns.regplot(x=x, y=y, scatter_kws=dict(alpha=0.10), line_kws=dict(color="red"))
plt.show()

Tout ça est prometteur, mais ici nous avons utilisé [`seaborn`](https://seaborn.pydata.org/) pour calculer les paramètres de l'hypothèse qui représente au mieux les données. Nous allons maintenant voir comment le faire nous même.

### Estimation des paramètres

Pour estimer $a$ et $b$, nous avons besoin de 2 éléments :

- une fonction de coût, qui nous dira pour des paramètres donnés $a$ et $b$ si l'on se débrouille bien ou non
- une méthode d'estimation des meilleurs paramètres étant donnée cette fonction de coût

### Prédictions

En prérequis au calcul de la fonction de coût, nous devons savoir calculer des prédictions.

Pour appliquer la transformation linéaire définie par $a$ et $b$ à $\mathbf{x}$, on utilise simplement la multiplication et l'addition de scalaires à un vecteur : $a\mathbf{x} + b$

*Définissez la fonction `predict` pour qu'elle calcule la transformation linéaire donnée par $a$ & $b$.*

In [ ]:
def predict(x: np.ndarray, a: float, b: float) -> np.ndarray:
    pass  # Votre code ici

#### Solution

In [ ]:
def predict(x: np.ndarray, a: float, b: float) -> np.ndarray:
    return a * x + b


x_example = np.array([1, 2, 3])
y_example = np.array([4, 10, 3])
a_example = 3
b_example = 2

print(predict(x_example, a_example, b_example))

### Fonction de coût

La fonction de coût la plus utilisée est la moyenne des différences entre les points réels et ceux obtenus par l'hypothèse avec les paramètres courants, le tout au carré.

En notant $E$ la fonction de coût, $r$ le nombre d'exemples d'apprentissage, on peut écrire :

$$
  E(a, b) = \sum\frac{(a\mathbf{x} + b - \mathbf{y})^2}{2r}
$$

- *Définissez la fonction `residuals` en vous aidant de la fonction `predict`*

- *Définissez la fonction `cost` en vous aidant de la fonction `residuals`*

In [ ]:
def residuals(
    x: np.ndarray,
    y: np.ndarray,
    a: float,
    b: float,
) -> np.ndarray:
    pass  # Votre code ici


def cost(
    x: np.ndarray,
    y: np.ndarray,
    a: float,
    b: float,
) -> np.float32:
    pass  # Votre code ici

#### Solution

In [ ]:
def residuals(
    x: np.ndarray,
    y: np.ndarray,
    a: float,
    b: float,
) -> np.ndarray:
    return predict(x, a, b) - y


print("Residuals:", residuals(x_example, y_example, a_example, b_example))


def cost(
    x: np.ndarray,
    y: np.ndarray,
    a: float,
    b: float,
) -> np.float32:
    return np.mean(residuals(x, y, a, b) ** 2) / 2


print("Costs:", cost(x_example, y_example, a_example, b_example))

### Optimisation des paramètres

Optimiser les paramètres revient à minimiser la fonction de coût :

$$\min_{a, b} \sum\frac{(a\mathbf{x} + b - \mathbf{y})^2}{2r}$$

Pour ce faire, il existe une formule directe que nous n'utiliserons pas dans ces travaux pratiques pour deux raisons : elle n'est pas applicable aux très grands datasets et la méthode que l'on va utiliser pourra être réutilisée pour la régression logistique et d'autres techniques (réseaux de neurones généraux, gradient boosted trees, etc).

Nous allons utiliser la descente de gradient. Cette méthode est itérative et fait des pas successifs en suivant la dérivée (pour maximiser) ou son opposé (pour minimiser) la fonction de perte. Dans le cas de la régression linéaire, elle converge vers l'optimum global (la meilleure solution).

En pseudo-code, l'algorithme est (avec les bons gradients calculés):

$$
\begin{aligned}
& \text{tant que ça n'a pas convergé :} \\
& \quad a \leftarrow a - \alpha \frac{\mathbf{x}^T(a\mathbf{x} + b - \mathbf{y})}{r} \\
& \quad b \leftarrow b - \alpha \sum\frac{a\mathbf{x_i} + b - \mathbf{y_i}}{r} \\
\end{aligned}
$$

où $\alpha$ est le pas d'apprentissage.


*Implémentez la fonction `gradient` en vous aidant de la fonction `residuals`.*

In [ ]:
def gradient(x: np.ndarray, y: np.ndarray, a: float, b: float) -> tuple[float, float]:
    pass  # Votre code ici

#### Solution

In [ ]:
def gradient(x: np.ndarray, y: np.ndarray, a: float, b: float) -> tuple[float, float]:
    r = residuals(x, y, a, b)
    g_a = (x @ r) / y.size
    g_b = r.mean()
    return g_a, g_b

### Descente de gradient

*Implémentez la descente de gradient dans la fonction `gradient_descent`.*

In [ ]:
def gradient_descent(
    x: np.ndarray, y: np.ndarray, alpha: float, nb_iter: int, epsilon: float
) -> tuple[float, float, typing.List[float]]:
    pass  # Votre code ici

#### Solution

In [ ]:
def gradient_descent(
    x: np.ndarray,
    y: np.ndarray,
    alpha: float = 0.1,
    nb_iter: int = 1000,
    epsilon: float = 10e-6,
) -> tuple[float, float, list[float]]:
    a = random.random()
    b = random.random()
    print(f"Paramètres initiaux (aléatoires) : a = {a:.2f}, b = {b:.2f}")
    initial_cost = cost(x, y, a, b)
    print(f"Coût initial : {initial_cost:.2f}")
    costs = [initial_cost]
    for i in range(nb_iter):
        g_a, g_b = gradient(x, y, a, b)
        a -= alpha * g_a
        b -= alpha * g_b
        costs.append(cost(x, y, a, b))
        if costs[-2] - costs[-1] < epsilon:
            print(f"Arrêt de la procédure : pas de progrès suffisant à l'itération {i}")
            break
    print(f"Paramètres finaux : a = {a:.2f}, b = {b:.2f}")
    print(f"Coût final : {costs[-1]:.2f}")
    return a, b, costs

Pour finir l'appel à la fonction gradient_descent

In [ ]:
a, b, costs = gradient_descent(x, y, 0.1, 1000)
print(f"Coûts successifs : {', '.join(map(str, costs))}")

### Vérification de notre hypothèse

Vérifions maintenant le modèle appris avec quelques plots :

- valeurs prédites contre les valeurs réelles
- résiduels
- coûts d'entrainement

In [ ]:
inv_x = x_scaler.inverse_transform(x[:, None])[:, 0]
inv_y = y_scaler.inverse_transform(y[:, None])[:, 0]
inv_yh = y_scaler.inverse_transform(predict(x, a, b)[:, None])[:, 0]

xs = np.arange(0, 6_000, 100)[:, None]
ys = y_scaler.inverse_transform(predict(x_scaler.transform(xs)[:, 0], a, b)[:, None])
plt.plot(inv_x, inv_y, "b.", alpha=0.10)
plt.plot(xs, ys, "r")
plt.title("Prédictions et valeurs réelles")
plt.xlabel("Surface habitable")
plt.ylabel("Prix de vente")
plt.show()

plt.plot(inv_x, inv_yh - inv_y, "r.", alpha=0.10)
plt.title("Résiduels")
plt.xlabel("Surface habitable")
plt.ylabel("Résiduels")
plt.show()

plt.plot(range(len(costs)), costs)
plt.title("Coûts d'apprentissage pendant la descente de gradient")
plt.xlabel("Itérations")
plt.ylabel("Erreur")
plt.show()

### Correction de l'asymétrie positive

In [ ]:
sns.distplot(train_df[["SalePrice"]], fit=scipy.stats.norm)
plt.title("Distribution de SalePrice avant normalisation")
plt.show()

### Standardize Y
Y_scaler_log = sklearn.preprocessing.StandardScaler()
Y = Y_scaler_log.fit_transform(np.log1p(train_df[["SalePrice"]]))[:, 0]

sns.distplot(Y, fit=scipy.stats.norm)
plt.title("Distribution de SalePrice après normalisation")
plt.show()

In [ ]:
a, b, costs = gradient_descent(x, Y, 0.01, 5000)

In [ ]:
inv_X = x_scaler.inverse_transform(x[:, None])[:, 0]
inv_Y = np.expm1(Y_scaler_log.inverse_transform(y[:, None])[:, 0])
inv_Yh = np.expm1(Y_scaler_log.inverse_transform(predict(x, a, b)[:, None])[:, 0])

Xs = np.arange(0, 6_000, 100).reshape(-1, 1)
Ys = np.expm1(Y_scaler_log.inverse_transform(predict(x_scaler.transform(Xs), a, b)))

In [ ]:
plt.plot(inv_X, inv_Y, "b.", alpha=0.10)
plt.plot(Xs, Ys, "r")
plt.title("Prédictions et valeurs réelles")
plt.xlabel("Surface habitable")
plt.ylabel("Prix de vente")
plt.show()

plt.plot(inv_X, inv_Yh - inv_Y, "r.", alpha=0.10)
plt.title("Résiduels")
plt.xlabel("Surface habitable")
plt.ylabel("Résiduels")
plt.show()

plt.plot(range(len(costs)), costs)
plt.title("Coûts d'apprentissage pendant la descente de gradient")
plt.xlabel("Itérations")
plt.ylabel("Erreur")
plt.show()

## Régression linéaire multivariée

Pour poursuivre ces travaux pratiques, nous allons reprendre un prétraitement complet effectué sur la totalité des données AMES (et non pas seulement la colonne `GrLiveArea`).

In [ ]:
!pip install nevergrad
import nevergrad

In [ ]:
def preprocess(train_file: str, test_file: str) -> np.ndarray:
    train_X = pd.read_csv(train_file, index_col="Id")
    test_X = pd.read_csv(test_file, index_col="Id")

    train_Y = train_X[["SalePrice"]]
    train_X = train_X.drop(columns=["SalePrice"])

    all_X = pd.concat([train_X, test_X])

    # Fill with median
    cols_1 = ["LotFrontage"]
    all_X[cols_1] = all_X[cols_1].fillna(train_X[cols_1].median())

    # Fill with mode
    cols_2 = [
        "MSZoning",
        "Electrical",
        "KitchenQual",
        "Exterior1st",
        "Exterior2nd",
        "SaleType",
        "Utilities",
    ]
    all_X[cols_2] = all_X[cols_2].fillna(train_X[cols_2].mode().iloc[0, :])

    # Fill with 0
    cols_4 = [
        "GarageYrBlt",
        "GarageArea",
        "GarageCars",
        "BsmtFinSF1",
        "BsmtFinSF2",
        "BsmtFullBath",
        "BsmtHalfBath",
        "BsmtUnfSF",
        "MasVnrArea",
        "TotalBsmtSF",
    ]
    all_X[cols_4] = all_X[cols_4].fillna(0)

    # Other fills
    cols_5 = ["Functional"]
    all_X[cols_5] = all_X[cols_5].fillna("Typ")

    # On donne à tous les autres NAs la valeur string NA, qui sera une catégorie
    all_X = all_X.fillna("NA")

    # On transforme le codage numérique en string afin que ce soit traité comme
    # une variable catégorielle
    cols_numerical2label = ["MSSubClass"]
    all_X[cols_numerical2label] = all_X[cols_numerical2label].astype(str)

    quality_mapping = dict(NA=0, Po=1, Fa=2, TA=3, Gd=4, Ex=5)
    quality_columns = [
        "BsmtCond",
        "BsmtQual",
        "ExterCond",
        "ExterQual",
        "FireplaceQu",
        "GarageCond",
        "GarageQual",
        "HeatingQC",
        "KitchenQual",
        "PoolQC",
    ]
    street_mapping = dict(NA=0, Grvl=1, Pave=2)
    bsmt_fin_mapping = dict(NA=0, Unf=1, LwQ=2, Rec=3, BLQ=4, ALQ=5, GLQ=6)

    replace_mapping = dict(
        Alley=street_mapping,
        BsmtExposure=dict(NA=0, No=1, Mn=2, Av=3, Gd=4),
        BsmtFinType1=bsmt_fin_mapping,
        BsmtFinType2=bsmt_fin_mapping,
        Functional=dict(Sal=1, Sev=2, Maj2=3, Maj1=4, Mod=5, Min2=6, Min1=7, Typ=8),
        LandSlope=dict(Sev=1, Mod=2, Gtl=3),
        LotShape=dict(IR3=1, IR2=2, IR1=3, Reg=4),
        PavedDrive=dict(NA=0, N=1, P=2, Y=3),
        Street=dict(Grvl=1, Pave=2),
        Utilities=dict(ELO=1, NoSeWa=2, NoSewr=3, AllPub=4),
    )

    for quality_column in quality_columns:
        replace_mapping[quality_column] = quality_mapping

    all_X.replace(replace_mapping, inplace=True)

    print(f"Nombre de NAs : {all_X.isnull().sum().sum()}")

    dummies = pd.get_dummies(all_X)
    return (
        dummies.iloc[: train_X.shape[0], :].values,
        train_Y.values,
        dummies.iloc[train_X.shape[0] :, :].values,
        dummies.columns,
    )

In [ ]:
X_train, Y_train, X_test, columns = preprocess(
    "dataset-ames/train.csv", "dataset-ames/test.csv"
)

### Normalisation de la variable de sortie

Utilisez [`sklearn.preprocessing.StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) pour centrer et réduire les cibles dans la nouvelle variable `Y_train_scaled`.

In [ ]:
# Votre code ici

#### Solution

In [ ]:
Y_scaler = sklearn.preprocessing.StandardScaler()
Y_train_scaled = Y_scaler.fit_transform(Y_train)

### Entraînement d'un modèle de régression linéaire simple

- Utilisez [`sklearn.linear_model.LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) pour apprendre une régression linéaire sur les données centrées et réduites.
- Affichez les coefficients de la régression apprise

In [ ]:
# Votre code ici

#### Solution

In [ ]:
linear_regression = sklearn.linear_model.LinearRegression()
linear_regression.fit(X_train, Y_train_scaled)
print(linear_regression.coef_)

In [ ]:
Y_scaler.inverse_transform(linear_regression.predict(X_train[:2]))

### Validation croisée

Utilisez une validation croisée définie avec [`sklearn.model_selection.cross_val_score`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html) sur un modèle de régression linéaire et vos données standardisées.

In [ ]:
# Votre code ici

#### Solution

In [ ]:
def score(
    model: sklearn.base.BaseEstimator,
    X: np.ndarray = X_train,
    Y: np.ndarray = Y_train_scaled,
):
    scores = sklearn.model_selection.cross_val_score(model, X, Y, cv=5, scoring="r2")
    return sum(scores) / len(scores)


def score(
    model: sklearn.base.BaseEstimator, X: np.ndarray = X_train, Y: np.ndarray = Y_train
) -> float:
    pipeline = sklearn.compose.TransformedTargetRegressor(
        regressor=model, transformer=sklearn.preprocessing.StandardScaler()
    )
    scores = sklearn.model_selection.cross_val_score(pipeline, X, Y, cv=5, scoring="r2")
    return sum(scores) / len(scores)


score(sklearn.linear_model.LinearRegression())

### Régularisation L1 et L2

Entraînez maintenant des modèles avec régularisation L1, L2, puis les deux ensemble.

In [ ]:
# Votre code ici

#### Solution

In [ ]:
score_l1 = score(sklearn.linear_model.Lasso())
score_l2 = score(sklearn.linear_model.Ridge())
score_l1_l2 = score(sklearn.linear_model.ElasticNet())
print(f"Scores l1, l2 et l1+l2 :", score_l1, score_l2, score_l1_l2)

### Recherche d'hyper-paramètres

`scikit-learn` contient quelques algorithmes de recherche d'hyperparamètres, mais ceux-ci sont assez limités (random search & grid search). Plusieurs librairies tierces sont disponibles pour fournir cette fonctionnalité.

- *Implémentez une fonction qui entraînera un modèle étant donné des hyper-paramètres et qui renverra un score qui sera minimal pour votre meilleur modèle*
- *Pour aller plus loin : utilisez comme dans un des exercices précédents une pipeline pour effectuer les prétraitements en isolation sur chaque bloc de la validation croisée.*
- *Pour aller encore plus loin : [Nevergrad](https://facebookresearch.github.io/nevergrad/), une librairie proposée par Facebook AI Research. Utilisez Nevergrad pour trouver les paramètres optimaux étant donné une fonction de perte que vous aurez défini comme l'opposé de la fonction score.*

In [ ]:
# Votre code ici

#### Solution

In [ ]:
# Modèle simple
model = sklearn.linear_model.Ridge()
params = dict(alpha=[1e-3, 1e-2, 1e-1, 1, 1e2, 1e3])
rscv = sklearn.model_selection.RandomizedSearchCV(
    model, params, scoring="r2", n_iter=6, verbose=0
)
search = rscv.fit(X_train, Y_train_scaled)
print(f"Paramètres trouvés par RSCV : {search.best_params_}")
print(f"score: {search.best_score_}")
print("=" * 80)

# Modèle complexe qui utilise une transformation des cibles
model = sklearn.compose.TransformedTargetRegressor(
    regressor=sklearn.linear_model.Ridge(),
    transformer=sklearn.preprocessing.StandardScaler(),
)
params = dict(regressor__alpha=[1e-3, 1e-2, 1e-1, 1, 1e2, 1e3])
rscv = sklearn.model_selection.RandomizedSearchCV(
    model, params, scoring="r2", n_iter=6, verbose=0
)
search = rscv.fit(X_train, Y_train)
print(f"Paramètres trouvés par RSCV : {search.best_params_}")
print(f"score: {search.best_score_}")
print("=" * 80)


# Avec Nevergrad
def loss(alpha: float) -> float:
    return -score(sklearn.linear_model.Ridge(alpha=alpha))


parametrization = nevergrad.p.Instrumentation(
    alpha=nevergrad.p.Scalar(lower=1e-3, upper=1e3),
)

optimizer = nevergrad.optimizers.NGOpt(parametrization=parametrization, budget=50)
recommendation = optimizer.minimize(loss, verbosity=0)

best_model = sklearn.linear_model.Ridge(**recommendation.kwargs)
best_model.fit(X_train, Y_train_scaled)
print(f"Paramètres trouvés par Nevergrad : {recommendation.kwargs}")
print(f"score: {score(best_model)}")

### Importance des caractéristiques

Utilisez la mesure d'importance par permutation de scikit-learn ([`sklearn.inspection.permutation_importance`](https://scikit-learn.org/stable/modules/generated/sklearn.inspection.permutation_importance.html)) pour évaluer quelles caractéristiques sont les plus utiles au meilleur modèle entraîné après la recherche d'hyper-paramètres.

In [ ]:
# Votre code ici

#### Solution

In [ ]:
feature_importances = sklearn.inspection.permutation_importance(
    best_model, X_train, Y_train_scaled, n_repeats=50
)

In [ ]:
series = pd.Series(feature_importances.importances_mean, index=columns)
series = series.sort_values(ascending=False).iloc[:10]
series.plot.bar()

### Caractéristiques polynômiales

Outre le travail d'extension manuel des caractéristiques, il est possible d'utiliser scikit-learn pour étendre la matrice d'apprentissage avec des caractéristiques polynômiales et sa classe [`sklearn.preprocessing.PolynomialFeatures`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html).

- *Étendez les matrices `X_train` et `X_test` restreintes aux 10 caractéristiques les plus importantes avec des features polynômiales pour créer `X_train_poly` et `X_test_poly`*
- *Entraînez un modèle régularisé L1 et L2 sur ces données*

In [ ]:
# Votre code ici

#### Solution

Sélection des 10 colonnes les plus importantes selon la permutation de lignes :

In [ ]:
top10 = (-feature_importances.importances_mean).argsort()[:10]
X_train_top10 = X_train[:, top10]
X_test_top10 = X_test[:, top10]

Création des colonnes polynômiales :

In [ ]:
def poly_features(array: np.ndarray) -> np.ndarray:
    polynomial_features = sklearn.preprocessing.PolynomialFeatures(
        2, interaction_only=True
    )
    return polynomial_features.fit_transform(array)


X_train_poly = poly_features(X_train_top10)
X_test_poly = poly_features(X_test_top10)

Une première version qui fait appel à RandomSearchCV :

In [ ]:
ridge = sklearn.linear_model.Ridge()

# Paramètre(s) à régler
distributions = dict(alpha=scipy.stats.uniform(loc=0, scale=50))

# Appel de la procédure qui règle le(s) paramètre(s)
clf = sklearn.model_selection.RandomizedSearchCV(ridge, distributions)
search = clf.fit(X_train_poly, Y_train_scaled)

# Exploitation du résultat
search.best_score_

Une deuxième version qui fait appel à la librairie Nevergrad.

In [ ]:
parametrization = nevergrad.p.Instrumentation(
    alpha=nevergrad.p.Scalar(lower=1e-3, upper=1e3),
)

optimizer = nevergrad.optimizers.NGOpt(parametrization=parametrization, budget=50)
recommendation = optimizer.minimize(loss, verbosity=0)

best_model = sklearn.linear_model.Ridge(**recommendation.kwargs)

print(f"Paramètres trouvés par Nevergrad : {recommendation.kwargs}")
print(f"score: {score(best_model, X=X_train_poly)}")